# 5-Minute Quickstart Guide to DuckPD

Welcome to the **DuckPD Quickstart Guide**! This notebook demonstrates how you can run fast, scalable **pandas-like data science workflows directly on top of DuckDB** without managing separate database clusters, servers, or proprietary accounts.

### What is DuckPD?
**DuckPD** brings a lazy, relational pandas-shaped API to the Python ecosystem with DuckDB as the high-performance analytical execution backend. You get the intuitive, familiar DataFrame and Series APIs you love, while execution benefits from:
- **Predicate and projection pushdown**
- **Vectorized multithreaded SQL execution**
- **Bounded Python-side materialization**
- **Zero silent full-frame fallback to pandas**

## Step 1: Import DuckPD

To get started, simply import `duckpd as pd`. Unlike other systems, DuckPD requires **no API keys, no cloud signups, and no cloud warehouse configurations**.

In [ ]:
from pathlib import Path

import duckpd as pd

print(f"DuckPD version: {pd.__version__}")

## Step 2: Establish a Session

Connect to an embedded DuckDB session. You can customize memory limits, thread pools, and temporary spill directories easily.

In [ ]:
session = pd.connect(memory_limit="512MB", threads=4)
print(f"Connected to DuckDB backend. Active executions: {session.execution_count}")

## Step 3: Load Data Directly via DuckDB

We load the classic **Goodreads Books dataset** directly from the web using DuckDB's automatic CSV scanner. DuckPD creates a **lazy plan**, preserves the file's physical row order automatically, and loads no data rows into Python memory yet.

In [ ]:
# Use DuckPD's public lazy CSV reader; no SQL wrapper is required.
df = session.read_csv("https://raw.githubusercontent.com/ponder-org/ponder-datasets/main/books.csv")

print("Lazy DataFrame:")
print(df)
print(f"\nColumns: {df.columns}")
print(f"Total element count (size): {df.size}")

## Step 4: Preview Data (Bounded Eager Head)

Calling `head()` executes a bounded preview with `LIMIT` pushed down into the database, returning only the requested rows into Python.

In [ ]:
df[["bookID", "title", "authors", "average_rating", "num_pages", "ratings_count"]].head(5)

## Step 5: Column Reductions & Summary Statistics

Calculate summary statistics on the numerical columns. Each reduction computes in a single, fast SQL query in DuckDB.

In [ ]:
num_cols = ["average_rating", "num_pages", "ratings_count", "text_reviews_count"]

print("--- Non-null counts ---")
print(df[num_cols].count())

print("\n--- Means ---")
print(df[num_cols].mean())

print("\n--- Minima ---")
print(df[num_cols].min())

print("\n--- Maxima ---")
print(df[num_cols].max())

## Step 6: Vectorized String Operations (`.str` accessor)

Use pandas-compatible string methods executed natively in DuckDB.

In [ ]:
# Filter Harry Potter books and standardize author names
hp_books = (
    df[df["title"].str.contains("Harry Potter")]
    .assign(
        author_upper=lambda f: f["authors"].str.upper(),
        title_len=lambda f: f["title"].str.len(),
    )[["bookID", "title", "author_upper", "average_rating", "num_pages"]]
    .sort_values("average_rating", ascending=False)
)

hp_books.head(5)

## Step 7: GroupBy & Aggregations

Perform analytical grouping across categories with named aggregations.

In [ ]:
language_summary = (
    df.groupby("language_code", as_index=False)
    .agg(
        total_books=("bookID", "size"),
        avg_rating=("average_rating", "mean"),
        total_pages=("num_pages", "sum"),
        max_reviews=("text_reviews_count", "max"),
    )
    .sort_values("total_books", ascending=False)
)

print("Top 10 Languages by Book Count:")
language_summary.head(10)

## Step 8: Inspecting Query Execution (`explain`)

DuckPD lets you inspect the full query plan, generated SQL, and DuckDB physical pipeline at any time!

In [ ]:
print(language_summary.explain())

## Step 9: Persisting and Exporting Results

Persist a reusable lazy result inside DuckDB, preview it as pandas, or write it directly to Parquet and CSV without routing the full result through pandas.

In [ ]:
# Persist once for reuse inside the current DuckDB session.
cached_summary = language_summary.persist()

# A bounded preview returns pandas explicitly.
pandas_df = cached_summary.head(5)
print(type(pandas_df))
display(pandas_df)

# Direct DuckDB-backed file writes stay under one ignored artifact directory.
demo_directory = Path("demo") if Path("demo").is_dir() else Path("..")
artifact_directory = demo_directory / ".artifacts"
artifact_directory.mkdir(exist_ok=True)
parquet_output = artifact_directory / "language_summary.parquet"
csv_output = artifact_directory / "language_summary.csv"
cached_summary.write_parquet(parquet_output, overwrite=True)
cached_summary.write_csv(csv_output)
print(f"\nExported {parquet_output} and {csv_output} via DuckDB!")